In [1]:
!pip install -q faiss-cpu sentence-transformers pypdf tqdm langchain langchain-experimental langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import faiss
import numpy as np
from tqdm import tqdm
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# =====================================
# CONFIGURATION
# =====================================
SOURCE_BASE = "/content/drive/MyDrive/CB/Nancy"
# Saving to a new folder so we don't overwrite your old data
TARGET_BASE = "/content/drive/MyDrive/CB/Nancy_Master_Index"

EMBED_MODEL = "all-MiniLM-L6-v2"

print("Loading embedding models...")
# Model 1: For the final FAISS vector database (with GPU batching enabled)
embed_model = SentenceTransformer(EMBED_MODEL)

# Model 2: The LangChain wrapper so the Semantic Chunker can use the same model
lc_embed_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Initialize the Semantic Chunker
print("Initializing Semantic Chunker (This ensures clinical thoughts stay intact)...")
text_splitter = SemanticChunker(
    lc_embed_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

# =====================================
# MASTER PROCESSING PIPELINE
# =====================================
def build_master_index():
    os.makedirs(TARGET_BASE, exist_ok=True)

    all_chunks = []
    all_metadata = []

    # 1. Gather all PDF paths from all subfolders
    pdf_paths = []
    for root, dirs, files in os.walk(SOURCE_BASE):
        for file in files:
            if file.endswith(".pdf"):
                pdf_paths.append(os.path.join(root, file))

    if not pdf_paths:
        print("No PDFs found in the source directory.")
        return

    print(f"Found {len(pdf_paths)} PDFs. Extracting and semantic-chunking text...")

    # 2. Extract and chunk text using LangChain Semantic Chunker
    for pdf_path in tqdm(pdf_paths, desc="Processing PDFs"):
        try:
            reader = PdfReader(pdf_path)
            text = ""
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted

            # Use the AI-driven Semantic Chunker to split the text based on meaning
            chunks = text_splitter.split_text(text)

            relative_folder = os.path.relpath(os.path.dirname(pdf_path), SOURCE_BASE)
            file_name = os.path.basename(pdf_path)

            for chunk in chunks:
                if len(chunk.strip()) > 50: # Ignore tiny junk chunks
                    all_chunks.append(chunk)
                    all_metadata.append({
                        "source_file": file_name,
                        "original_category": relative_folder
                    })

        except Exception as e:
            print(f"Error reading {pdf_path}: {e}")

    if len(all_chunks) == 0:
        print("No valid content could be extracted.")
        return

    # 3. Create Embeddings (Optimized with batch_size for GPU)
    print(f"\nCreating embeddings for {len(all_chunks)} chunks (Using GPU Batching)...")
    embeddings = embed_model.encode(
        all_chunks,
        batch_size=128,          # 🌟 SPEED UP: Processes 128 chunks at once
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")

    # 4. Build single FAISS Index
    print("Building master FAISS index...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # 5. Save everything to the new TARGET_BASE
    faiss.write_index(index, os.path.join(TARGET_BASE, "master_index.faiss"))

    with open(os.path.join(TARGET_BASE, "master_chunks.json"), "w") as f:
        json.dump(all_chunks, f)

    with open(os.path.join(TARGET_BASE, "master_metadata.json"), "w") as f:
        json.dump(all_metadata, f)

    print("\n✅ Master Semantic Index built and saved successfully!")

# Run the pipeline
build_master_index()

# =====================================
# VERIFICATION
# =====================================
print("\nVerifying saved files in new folder:")
for file in os.listdir(TARGET_BASE):
    if file.startswith("master_"):
        file_path = os.path.join(TARGET_BASE, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"→ {file} ({size_mb:.2f} MB)")

Mounted at /content/drive
Loading embedding models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing Semantic Chunker (This ensures clinical thoughts stay intact)...
Found 46 PDFs. Extracting and semantic-chunking text...


Processing PDFs:  74%|███████▍  | 34/46 [20:16<32:59, 164.95s/it]

In [ ]:
%%writefile app.py

import streamlit as st
import faiss
import json
import numpy as np
import torch
import re
import os
import gc  # 🌟 Added for Garbage Collection to prevent freezing

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# =====================================================
# PAGE SETTINGS & CONFIG
# =====================================================
st.set_page_config(page_title="🧠 Personal Mental Health Chatbot")
st.title("🧠 Personal Mental Health Chatbot")

# 🌟 Tell the app to use the T4 GPU if it's turned on!
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 🌟 Increased threshold since we are searching all PDFs at once
RELEVANCE_THRESHOLD = 0.65
TOP_K = 3

# 🌟 Pointing to the new Master Index folder
BASE_INDEX_PATH = "/content/drive/MyDrive/CB/Nancy_Master_Index"
if not os.path.exists(BASE_INDEX_PATH):
    st.error("Nancy_Master_Index folder not found. Please run the data pipeline first.")
    st.stop()

# =====================================================
# LOAD MODELS
# =====================================================
@st.cache_resource
def load_embedding():
    return SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

@st.cache_resource
def load_llm():
    # If you want to upgrade to a smarter model later, change this name
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

embedding_model = load_embedding()
tokenizer, model = load_llm()

# =====================================================
# LOAD MASTER INDEX
# =====================================================
@st.cache_resource
def load_master_index():
    faiss_path = os.path.join(BASE_INDEX_PATH, "master_index.faiss")
    chunks_path = os.path.join(BASE_INDEX_PATH, "master_chunks.json")

    if not os.path.exists(faiss_path) or not os.path.exists(chunks_path):
        return None, None

    index = faiss.read_index(faiss_path)
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    return index, chunks

def extract_text(chunk):
    if isinstance(chunk, str):
        return chunk
    if isinstance(chunk, dict) and "text" in chunk:
        return chunk["text"]
    return ""

# =====================================================
# RETRIEVAL (No Categories Needed!)
# =====================================================
def retrieve_best_chunk(query):
    index, chunks = load_master_index()

    if index is None:
        return None, 0.0

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        device=DEVICE
    ).astype("float32")

    D, I = index.search(query_embedding, TOP_K)

    selected_chunks = [
        extract_text(chunks[idx])
        for score, idx in zip(D[0], I[0])
        if idx >= 0 and score >= RELEVANCE_THRESHOLD
    ]

    if not selected_chunks:
        return None, float(D[0][0])

    return "\n\n".join(selected_chunks), float(D[0][0])

# =====================================================
# TEXT CLEANING & SPLITTING
# =====================================================
def clean_generated_text(text):
    text = re.sub(r'\*\*','',text)
    text = re.sub(r'\s+',' ',text)
    return text.strip()

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 30]

# =====================================================
# LLM GENERATION TASKS
# =====================================================
def generate_brief_summary(query):
    prompt = f"Summarize the emotional issue in the following question in 2 short sentences.\nUser question: {query}\nSummary:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.3)
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return clean_generated_text(summary)

def generate_motivation(query):
    prompt = f"""
You are a compassionate mental health guide.
Write a meaningful motivational reflection for someone facing the following emotional situation.
Important rules: Do NOT give instructions. Do NOT diagnose. Write in a warm tone. Focus on hope.
Emotional situation: {query}
Motivational reflection:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=180, temperature=0.75, top_p=0.92, repetition_penalty=1.15, do_sample=True
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    text = clean_generated_text(tokenizer.decode(generated_tokens, skip_special_tokens=True))
    text = re.sub(r'["“”]', "", text)
    text = re.sub(r'\d+\)', "", text)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 20]
    return "\n".join(sentences[:8])

GEN_KWARGS = dict(max_new_tokens=420, temperature=0.4, top_p=0.9, repetition_penalty=1.2, do_sample=True)

def generate_answer(query, context=None):
    system_message = (
        "You are a compassionate mental health support assistant.\n\n"
        "Rules:\n- Do not diagnose mental illness.\n- Explain emotional experiences clearly.\n"
        "- Use supportive language.\n- Provide helpful coping suggestions.\n"
    )

    if context:
        user_message = f"Reference material:\n{context}\n\nUser question:\n{query}\nExplain the emotional situation clearly."
    else:
        user_message = f"User question:\n{query}\nExplain the emotional challenge in supportive language."

    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(**inputs, **GEN_KWARGS)

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    brief_summary = generate_brief_summary(query)
    motivation = generate_motivation(query)

    return format_structured_answer(response, brief_summary, motivation)

def format_structured_answer(text, brief_summary, motivation):
    text = clean_generated_text(text)
    sentences = split_sentences(text)

    if len(sentences) < 6:
        return text

    intro = sentences[0]
    explanation = " ".join(sentences[1:6])
    key_points = sentences[6:9] if len(sentences) >= 9 else sentences[2:5]
    impact = sentences[9:12] if len(sentences) >= 12 else sentences[-4:-1]
    conclusion = sentences[-1]

    formatted = f"**Introduction**\n\n{intro}\n\n**Brief Explanation**\n\n{brief_summary}\n\n**Explanation**\n\n{explanation}\n\n**Key Points**\n"
    for kp in key_points: formatted += f"\n- {kp}"
    formatted += "\n\n**Impact and Coping Strategies**\n"
    for im in impact: formatted += f"\n- {im}"
    formatted += f"\n\n**Conclusion**\n\n{conclusion}\n\n**Motivation**\n\n{motivation}\n"

    return formatted.strip()

# =====================================================
# CHAT UI
# =====================================================
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

user_input = st.chat_input("Ask about stress, anxiety, sleep, depression...")

if user_input:
    st.session_state.chat_history.append({"role": "user", "content": user_input})

    with st.spinner("Thinking..."):
        chunk, score = retrieve_best_chunk(user_input)
        reply = generate_answer(user_input, chunk if chunk else None)
        source_type = "📖 RAG Knowledge Base" if chunk else "🤖 General Model Response"

    st.session_state.chat_history.append({"role": "assistant", "content": reply})

    with st.expander("🔎 Transparency Details"):
        st.write("Similarity Score:", round(score, 4))
        if chunk:
            st.write("Retrieved Context Preview:")
            st.write(chunk[:500] + "...")
        else:
            st.write("No relevant textbook chunk found. Generating from general knowledge.")
        st.write("Answer Source:", source_type)

    # 🌟 THE GARBAGE COLLECTOR (Keeps memory clean for fast follow-up questions)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for msg in st.session_state.chat_history:
    with st.chat_message("user" if msg["role"] == "user" else "assistant"):
        st.markdown(f"<span style='font-size:16px'>{msg['content']}</span>", unsafe_allow_html=True)

In [ ]:
!pip install streamlit pyngrok
!pip install pyngrok
!pkill -f streamlit
!streamlit run app.py &>/dev/null &
!pip install sentence-transformers faiss-cpu

In [ ]:

!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()
# Put your real ngrok key below!
ngrok.set_auth_token("39yHDphsAXGu68A4ZJHuvquoswI_6Jx1gd8aZ9ySJwXs1bgBY")
public_url = ngrok.connect(8501)
print("🌟 Your App URL:", public_url)

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0